# Circularity Normal Distribution Analysis

1. Read and calculate the circularity for each kernel.
2. Calculate `mean_circularity`.
3. Plot the overall distribution of `mean_circularity` for all photos.

In [ ]:
from pathlib import Path
import math
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

PIPELINE_DIR = Path(r"C:\Users\HP\Desktop\Academic Presentation\seed_project\a_pipeline_test")
CONFIG_PATH = PIPELINE_DIR / "config.yaml"

OUTPUT_DIR_OVERRIDE = None

MIN_KERNELS_PER_IMAGE = 1
BINS = 70
GRID_SIZE = 1200
MAX_SADDLE_MARKS = 6
KDE_BANDWIDTH = None
SAVE_DPI = 200

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else Path(config["output"]["base_dir"])
ANALYSIS_DIR = OUTPUT_DIR / "circularity_normal_distribution"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Pipeline dir:", PIPELINE_DIR)
print("Output dir  :", OUTPUT_DIR)
print("Analysis dir:", ANALYSIS_DIR)

In [ ]:
def get_original_image_name(kernel_name):
    """Recover original tray image name from a kernel subimage name."""
    base = os.path.basename(str(kernel_name))
    stem, ext = os.path.splitext(base)
    stem = re.sub(r"_round$", "", stem, flags=re.IGNORECASE)
    parts = stem.rsplit("_kernel_", 1)
    if len(parts) == 2:
        return parts[0] + ext
    return base


def compute_circularity(area, perimeter):
    """Circularity = 4*pi*A/P^2."""
    if perimeter <= 0:
        return np.nan
    return float((4.0 * math.pi * float(area)) / (float(perimeter) ** 2))


def load_from_measurements_csv(output_dir, config):
    measurements_path = output_dir / config["output"].get("measurements_csv", "measurements.csv")
    if not measurements_path.exists():
        return None, measurements_path

    df = pd.read_csv(measurements_path)
    if "kernel_name" not in df.columns:
        raise ValueError(f"{measurements_path} 缺少 kernel_name 列。")

    circularity = (
        pd.to_numeric(df["circularity"], errors="coerce")
        if "circularity" in df.columns
        else pd.Series(np.nan, index=df.index)
    )
    round_circularity = (
        pd.to_numeric(df["round_circularity"], errors="coerce")
        if "round_circularity" in df.columns
        else pd.Series(np.nan, index=df.index)
    )
    circularity = circularity.combine_first(round_circularity)

    out = pd.DataFrame({
        "kernel_name": df["kernel_name"].astype(str),
        "image_name": df["kernel_name"].map(get_original_image_name),
        "circularity": circularity,
        "source": "measurements_csv",
    })
    if "shape_label" in df.columns:
        out["shape_label"] = df["shape_label"].fillna("").astype(str)
    else:
        out["shape_label"] = ""
    return out, measurements_path


def compute_from_masks(output_dir, config):
    """Fallback path: compute circularity directly from masks_binary/."""
    import cv2

    masks_dir = output_dir / config["output"].get("masks_binary_dir", "masks_binary")
    if not masks_dir.exists():
        raise FileNotFoundError(f"not found measurements.csv，not found masks_binary: {masks_dir}")

    rows = []
    image_files = sorted([
        p for p in masks_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    ])
    min_area = float(config.get("contour_filtering", {}).get("min_contour_area", 500))

    for path in image_files:
        mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        area = float(cv2.contourArea(contour))
        if area < min_area:
            continue
        perimeter = float(cv2.arcLength(contour, True))
        rows.append({
            "kernel_name": path.name,
            "image_name": get_original_image_name(path.name),
            "circularity": compute_circularity(area, perimeter),
            "source": "masks_binary_fallback",
            "shape_label": "",
        })

    return pd.DataFrame(rows), masks_dir


def load_kernel_circularity(output_dir, config):
    kernel_df, source_path = load_from_measurements_csv(output_dir, config)
    if kernel_df is None:
        kernel_df, source_path = compute_from_masks(output_dir, config)

    kernel_df["circularity"] = pd.to_numeric(kernel_df["circularity"], errors="coerce")
    kernel_df = kernel_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["circularity"])
    kernel_df = kernel_df[(kernel_df["circularity"] > 0) & (kernel_df["circularity"] <= 1.05)].copy()
    return kernel_df, source_path

In [ ]:
kernel_df, source_path = load_kernel_circularity(OUTPUT_DIR, config)
print("Circularity source:", source_path)
print("Valid kernels     :", len(kernel_df))
print("Tray images       :", kernel_df["image_name"].nunique())

image_summary = (
    kernel_df
    .groupby("image_name", as_index=False)
    .agg(
        kernel_count=("circularity", "size"),
        mean_circularity=("circularity", "mean"),
        median_circularity=("circularity", "median"),
        std_circularity=("circularity", "std"),
        min_circularity=("circularity", "min"),
        max_circularity=("circularity", "max"),
    )
)
image_summary = image_summary[image_summary["kernel_count"] >= MIN_KERNELS_PER_IMAGE].copy()
image_summary["std_circularity"] = image_summary["std_circularity"].fillna(0.0)

summary_csv = ANALYSIS_DIR / "image_mean_circularity.csv"
image_summary.to_csv(summary_csv, index=False)

print("Images after kernel-count filter:", len(image_summary))
print("Saved image summary:", summary_csv)
display(image_summary.head())

In [ ]:
def silverman_bandwidth(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n <= 1:
        return 0.01
    std = float(np.std(values, ddof=1))
    q75, q25 = np.percentile(values, [75, 25])
    iqr_sigma = float((q75 - q25) / 1.349) if q75 > q25 else std
    sigma = min(std, iqr_sigma) if iqr_sigma > 0 else std
    if sigma <= 1e-9:
        sigma = max(float(np.std(values)), 0.01)
    return max(0.9 * sigma * (n ** (-1 / 5)), 1e-4)


def gaussian_kde_1d(values, grid, bandwidth=None, chunk_size=5000):
    values = np.asarray(values, dtype=float)
    grid = np.asarray(grid, dtype=float)
    bw = silverman_bandwidth(values) if bandwidth is None else float(bandwidth)
    bw = max(bw, 1e-4)

    density = np.zeros_like(grid, dtype=float)
    norm = 1.0 / (bw * math.sqrt(2.0 * math.pi))
    for start in range(0, len(values), chunk_size):
        chunk = values[start:start + chunk_size]
        diff = (grid[:, None] - chunk[None, :]) / bw
        density += np.exp(-0.5 * diff ** 2).sum(axis=1) * norm
    density /= max(len(values), 1)
    return density, bw


def find_saddle_candidates(grid, density):
    """Find 1D valley candidates between local peaks on the KDE curve."""
    grid = np.asarray(grid, dtype=float)
    density = np.asarray(density, dtype=float)
    if len(grid) < 5:
        return pd.DataFrame(columns=["threshold", "density", "relative_depth"])

    peaks = np.where((density[1:-1] > density[:-2]) & (density[1:-1] > density[2:]))[0] + 1
    valleys = np.where((density[1:-1] < density[:-2]) & (density[1:-1] < density[2:]))[0] + 1

    rows = []
    for valley_idx in valleys:
        left_peaks = peaks[peaks < valley_idx]
        right_peaks = peaks[peaks > valley_idx]
        if len(left_peaks) == 0 or len(right_peaks) == 0:
            continue
        left_peak = left_peaks[np.argmax(density[left_peaks])]
        right_peak = right_peaks[np.argmax(density[right_peaks])]
        lower_peak_density = min(float(density[left_peak]), float(density[right_peak]))
        valley_density = float(density[valley_idx])
        depth = max(0.0, lower_peak_density - valley_density)
        relative_depth = depth / max(lower_peak_density, 1e-12)
        rows.append({
            "threshold": float(grid[valley_idx]),
            "density": valley_density,
            "relative_depth": float(relative_depth),
            "left_peak_x": float(grid[left_peak]),
            "right_peak_x": float(grid[right_peak]),
            "left_peak_density": float(density[left_peak]),
            "right_peak_density": float(density[right_peak]),
        })

    candidates = pd.DataFrame(rows)
    if not candidates.empty:
        candidates = candidates.sort_values(["relative_depth", "density"], ascending=[False, True]).reset_index(drop=True)
    return candidates


values = image_summary["mean_circularity"].to_numpy(dtype=float)
if len(values) < 2:
    raise ValueError("There are fewer than 2 photos available for distribution analysis; therefore, a distribution plot cannot be generated.")

x_min = max(0.0, float(np.min(values)) - 0.02)
x_max = min(1.05, float(np.max(values)) + 0.02)
if x_max <= x_min:
    x_min, x_max = 0.0, 1.0

grid = np.linspace(x_min, x_max, GRID_SIZE)
kde_density, kde_bandwidth = gaussian_kde_1d(values, grid, bandwidth=KDE_BANDWIDTH)
saddle_candidates = find_saddle_candidates(grid, kde_density)

saddle_csv = ANALYSIS_DIR / "saddle_candidates.csv"
saddle_candidates.to_csv(saddle_csv, index=False)

print(f"KDE bandwidth: {kde_bandwidth:.6f}")
print("Saved saddle candidates:", saddle_csv)
display(saddle_candidates.head(MAX_SADDLE_MARKS))

In [ ]:
mu = float(np.mean(values))
sigma = float(np.std(values, ddof=1))
normal_density = None
if sigma > 1e-12:
    normal_density = (
        (1.0 / (sigma * math.sqrt(2.0 * math.pi)))
        * np.exp(-0.5 * ((grid - mu) / sigma) ** 2)
    )

fig, ax = plt.subplots(figsize=(12, 7))
ax.hist(
    values,
    bins=BINS,
    range=(x_min, x_max),
    density=True,
    color="goldenrod",
    alpha=0.42,
    edgecolor="white",
    linewidth=0.7,
    label="Per-image mean circularity histogram",
)
ax.plot(grid, kde_density, color="saddlebrown", linewidth=2.5, label=f"KDE empirical density (bw={kde_bandwidth:.4f})")
if normal_density is not None:
    ax.plot(grid, normal_density, color="steelblue", linewidth=2.0, linestyle="--", label=f"Normal fit: mean={mu:.4f}, std={sigma:.4f}")

ax.axvline(mu, color="black", linestyle=":", linewidth=1.8, label=f"Global mean={mu:.4f}")

top_candidates = saddle_candidates.head(MAX_SADDLE_MARKS)
colors = ["crimson", "orangered", "darkmagenta", "teal", "olive", "navy"]
for idx, row in top_candidates.iterrows():
    color = colors[idx % len(colors)]
    threshold = float(row["threshold"])
    density = float(row["density"])
    ax.axvline(threshold, color=color, linestyle="-.", linewidth=1.5, alpha=0.9)
    ax.scatter([threshold], [density], color=color, s=45, zorder=5)
    ax.text(
        threshold,
        density,
        f"  saddle {idx + 1}\n  {threshold:.4f}",
        color=color,
        fontsize=9,
        va="bottom",
    )

ax.set_title(
    "Distribution of Per-Image Mean Kernel Circularity\n"
    f"n_images={len(values):,}, n_kernels={len(kernel_df):,}"
)
ax.set_xlabel("Mean circularity per original tray image")
ax.set_ylabel("Density")
ax.grid(True, alpha=0.25)
ax.legend(loc="best", fontsize=9)
plt.tight_layout()

figure_path = ANALYSIS_DIR / "mean_circularity_distribution_with_saddles.png"
fig.savefig(figure_path, dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

print("Saved figure:", figure_path)

## How to Read This Chart

- `golden histogram`: The distribution of the average circularity for each original photo.
- `brown KDE`: Empirical distribution curve, better suited for identifying multiple peaks and valleys.
- `blue dashed normal fit`: A single-peak normal curve fitted using the average circularity of all photos, serving as a reference for the overall trend.
- `saddle candidates`: Candidate local troughs on the KDE curve; these are generally more useful than simply looking at the mean when determining thresholds.

Output files:

- `circularity_normal_distribution/image_mean_circularity.csv`
- `circularity_normal_distribution/saddle_candidates.csv`
- `circularity_normal_distribution/mean_circularity_distribution_with_saddles.png`

If the plot is too smooth and saddle points are not clearly visible, reduce the `KDE_BANDWIDTH` value, for example to `0.005` or `0.01`. If the plot is too jagged, increase the `KDE_BANDWIDTH` value.